# $(SASA) Models - Kmeans$

In [1]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import pickle
import numpy as np
import pandas as pd

from pmbrl.model2 import Base_Line_Simple_Model
# from pmbrl.model2 import Model, Regularized_Reference_Loss
from pmbrl.data import Experiment_Data, get_data_expanded, get_data_compacted

# Predict 

In [2]:
def optim_params(prediction_dataset):
    weights = prediction_dataset.copy()
    for m, _ in enumerate(models):
        weights['estimated_s'] = prediction_dataset[f'estimated_s_model_{m}']
        expansions = {f'estimated_s': [f'estimated_s0', f'estimated_s1', f'estimated_s2', f'estimated_s3']}
        ds = get_data_expanded(weights, expansions)
        # weights[f'estimated_s_model_{m}'] = ds['estimated_s']
        for d in range(4):
            weights[f'estimated_s{d}_model{m}'] = ds[f'estimated_s{d}']
    weights['A'] = weights.apply(lambda row: np.array([[row[f'estimated_s{d}_model{m}'] for m,_ in enumerate(models)] for d in range(4)]),axis=1)
    weights['b'] = weights.apply(lambda row: np.array([row[f's__{d}'] for d in range(4)]),axis=1)
    weights['w'] = weights.apply(lambda row: np.linalg.lstsq(row.A, row.b)[0],axis=1)

    weights[[f'estimated_weight_{m}' for m, _ in enumerate(models)]] = weights.apply(lambda row: pd.Series(row['w']),axis=1)
    return weights


In [3]:

def predict_with_params(prediction_dataset, weights):
    for m, _ in enumerate(models):
        prediction_dataset[f'weight_{m}'] = weights[f'estimated_weight_{m}']

    expansions = {
        f'estimated_s_model_{m}': [f'estimated_s{d}_model{m}' for d in range(4)] for m,_ in enumerate(models)}
    ds = get_data_expanded(prediction_dataset, expansions)
    
    for d in range(4):
        ds[f'estimated_weighted_s{d}'] = ds.apply(
            lambda row: np.sum([row[f'estimated_s{d}_model{m}']* row[f'weight_{m}'] for m, _ in enumerate(models)])
            , axis=1
        )
    
    ds = get_data_compacted(ds, {'estimated_weighted_s': [f'estimated_weighted_s{d}' for d in range(4)]})
     
    prediction_dataset['estimated_r'] = ds['estimated_r_model_0']
    prediction_dataset['estimated_s'] = ds['estimated_weighted_s']
    
    results = data.get_evaluation_metrics(prediction_dataset, p=False)
    prediction_dataset[f'rse'] = results['rse']
    prediction_dataset[f'rse_normalized'] = results['rse_normalized']

    prediction_dataset[f'rse_s0'] = results['rse_s0']
    prediction_dataset[f'rse_s1'] = results['rse_s1']
    prediction_dataset[f'rse_s2'] = results['rse_s2']
    prediction_dataset[f'rse_s3'] = results['rse_s3']

    prediction_dataset[f'rse_s0_normalized'] = results['rse_s0_normalized']
    prediction_dataset[f'rse_s1_normalized'] = results['rse_s1_normalized']
    prediction_dataset[f'rse_s2_normalized'] = results['rse_s2_normalized']
    prediction_dataset[f'rse_s3_normalized'] = results['rse_s3_normalized']

    return prediction_dataset

In [4]:
def evaluate(models, df):
    def predict(model):
        prediction_dataset = df.copy()
        prediction_dataset[model.grouped_targets_lables] = prediction_dataset.apply(lambda row: data._predict_from_row(row, model), axis=1, result_type='expand')
        return prediction_dataset

    predictions = [predict(m) for m in models]

    prediction_dataset = df.copy()
    for i, pred in enumerate(predictions):
        prediction_dataset[f'estimated_r_model_{i}'] = pred['estimated_r']
        prediction_dataset[f'estimated_s_model_{i}'] = pred['estimated_s']
        
        results = data.get_evaluation_metrics(pred, p=False)
        prediction_dataset[f'rse_model_{i}'] = results['rse']
        prediction_dataset[f'rse_normalized_model_{i}'] = results['rse_normalized']

        prediction_dataset[f'rse_s0_model_{i}'] = results['rse_s0']
        prediction_dataset[f'rse_s1_model_{i}'] = results['rse_s1']
        prediction_dataset[f'rse_s2_model_{i}'] = results['rse_s2']
        prediction_dataset[f'rse_s3_model_{i}'] = results['rse_s3']

        prediction_dataset[f'rse_s0_normalized_model_{i}'] = results['rse_s0_normalized']
        prediction_dataset[f'rse_s1_normalized_model_{i}'] = results['rse_s1_normalized']
        prediction_dataset[f'rse_s2_normalized_model_{i}'] = results['rse_s2_normalized']
        prediction_dataset[f'rse_s3_normalized_model_{i}'] = results['rse_s3_normalized']


    return prediction_dataset

In [5]:
def predicts(models, df):
    pre_df = df.copy()
    pre_df[['s_', 's_0', 's_1', 's_2', 's_3', 'a_', 's__0', 's__1', 's__2', 's__3', 's__']] = pre_df[['s', 's0', 's1', 's2', 's3', 'a', 's_0', 's_1', 's_2', 's_3', 's_']]

    prediction_dataset = evaluate(models, pre_df)
    params = optim_params(prediction_dataset)
    prediction_dataset = evaluate(models, df)
    final_predictions = predict_with_params(prediction_dataset, params)

    cols = [
        's__', 'estimated_s', 'rse', 'rse_normalized',
        'rse_s0', 'rse_s1', 'rse_s2', 'rse_s3', 
        'rse_s0_normalized', 'rse_s1_normalized',
        'rse_s2_normalized', 'rse_s3_normalized'
    ] + [f'weight_{m}' for m, _ in enumerate(models)] + [f'estimated_s_model_{m}' for m, _ in enumerate(models)]

    return final_predictions[cols]

In [6]:
rses, rses_norm = [], []
rse_s0, rse_s1,	rse_s2,	rse_s3 = [],[],[],[]

def store_data(r):
    rses.append(r.rse.mean())
    rses_norm.append(r.rse_normalized.mean())
    rse_s0.append(r.rse_s0.mean())
    rse_s1.append(r.rse_s1.mean())
    rse_s2.append(r.rse_s2.mean())
    rse_s3.append(r.rse_s3.mean())
    
for m in range(10):
    print('model: ', m)
    nome_do_arquivo = f'kmodels_{m+1}.pkl'

    with open(nome_do_arquivo, 'rb') as arquivo:
        exp = pickle.load(arquivo)
        data = exp['data']
        models = exp['model']

    del data
    del exp
    del arquivo

    data = Experiment_Data()
    data.load(path='../testing_data.csv')

    expansions = {
        's': ['s0', 's1', 's2', 's3'],
        's_': ['s_0', 's_1', 's_2', 's_3'],
        's__': ['s__0', 's__1', 's__2', 's__3'],
    }

    df = get_data_expanded(data.build_training_dataset(), expansions)
    prediction_dataset = predicts(models, df)
    
    store_data(prediction_dataset)


model:  0
model:  1
model:  2
model:  3
model:  4
model:  5
model:  6
model:  7
model:  8
model:  9


In [7]:
print('rse:', sum(rses)/len(rses))
print('rses_norm:', sum(rses_norm)/len(rses_norm))
print('rse_s0:', sum(rse_s0)/len(rse_s0))
print('rse_s1:', sum(rse_s1)/len(rse_s1))
print('rse_s2:', sum(rse_s2)/len(rse_s2))
print('rse_s3:', sum(rse_s3)/len(rse_s3))

rse: 0.13191928499913708
rses_norm: 0.509385003487808
rse_s0: 0.0048879475768859254
rse_s1: 0.013469792518866444
rse_s2: 0.004968108908170369
rse_s3: 0.10859343599521434
